# Lumis 02: Clean and split

Builds the Lumis dataset from DermNet (CP1 section 3.5.1):

1. Keeps only the folders for the five Lumis classes and pools DermNet's own train and test images
2. Drops unreadable and tiny images
3. Removes exact duplicates, and drops images whose duplicates carry a different class
4. Groups near-duplicates (perceptual hash) so they always land in the same split, which prevents train/test leakage
5. Makes a 70/15/15 train/validation/test split per class with a fixed seed
6. Saves the split manifest, the cleaning log, and review images to Drive

**The saved split is the frozen test set.** If `lumis_split_v1.csv` already exists in Drive, this notebook stops instead of overwriting it. To redo the split deliberately, rename that file first.

Runtime: CPU is fine for this notebook.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, zipfile
BASE = '/content/drive/MyDrive/Lumis'
ZIP = f'{BASE}/data/dermnet.zip'
LOCAL = '/content/dermnet'
SPLIT_FILE = f'{BASE}/data/lumis_split_v1.csv'
SEED = 42
os.makedirs(f'{BASE}/exports', exist_ok=True)

if os.path.exists(SPLIT_FILE):
    raise RuntimeError(f'{SPLIT_FILE} already exists: the test set is frozen. Rename it to redo the split.')

if not os.path.exists(LOCAL):
    with zipfile.ZipFile(ZIP) as z:
        z.extractall(LOCAL)
print('Ready')

## 2. Map DermNet folders to the five Lumis classes

In [ ]:
from pathlib import Path
import pandas as pd

CLASS_MAP = {
    'Eczema Photos': 'eczema',
    'Atopic Dermatitis Photos': 'eczema',
    'Psoriasis pictures Lichen Planus and related diseases': 'psoriasis',
    'Tinea Ringworm Candidiasis and other Fungal Infections': 'tinea',
    'Acne and Rosacea Photos': 'acne',
    'Scabies Lyme Disease and other Infestations and Bites': 'scabies',
}
IMG_EXT = {'.jpg', '.jpeg', '.png'}

rows = []
split_dirs = [p for p in Path(LOCAL).rglob('*') if p.is_dir() and p.name.lower() in ('train', 'test')]
for split_dir in split_dirs:
    for folder, cls in CLASS_MAP.items():
        d = split_dir / folder
        assert d.exists(), f'Missing folder: {d}'
        for f in d.iterdir():
            if f.suffix.lower() in IMG_EXT:
                rows.append({'path': str(f.relative_to(LOCAL)), 'source_folder': folder,
                             'kaggle_split': split_dir.name, 'lumis_class': cls})

df = pd.DataFrame(rows)
log = {'images_in_selected_folders': len(df)}
print(len(df), 'images')
print(df.lumis_class.value_counts())

## 3. Check every image and compute hashes

MD5 finds byte-identical files; the perceptual hash (pHash) finds visually near-identical images such as resized or re-saved copies. Takes a few minutes.

In [ ]:
!pip install -q imagehash
import hashlib, imagehash
import numpy as np
from PIL import Image
from tqdm.auto import tqdm

MIN_SIDE = 100
recs = []
for p in tqdm(df.path):
    r = {'path': p, 'readable': True, 'width': None, 'height': None, 'md5': None, 'phash': None}
    try:
        fp = Path(LOCAL) / p
        r['md5'] = hashlib.md5(fp.read_bytes()).hexdigest()
        with Image.open(fp) as im:
            im = im.convert('RGB')
            r['width'], r['height'] = im.size
            r['phash'] = str(imagehash.phash(im))
    except Exception:
        r['readable'] = False
    recs.append(r)
df = df.merge(pd.DataFrame(recs), on='path')
print('Unreadable:', int((~df.readable).sum()))

## 4. Drop unreadable, tiny, and exact-duplicate images

In [ ]:
bad = ~df.readable
log['dropped_unreadable'] = int(bad.sum())
df = df[~bad].copy()

tiny = df[['width', 'height']].min(axis=1) < MIN_SIDE
log['dropped_smaller_than_100px'] = int(tiny.sum())
df = df[~tiny].copy()

# The same file filed under two different Lumis classes: the label is unreliable, drop every copy
n_cls = df.groupby('md5').lumis_class.nunique()
conflict = df.md5.isin(n_cls[n_cls > 1].index)
log['dropped_exact_duplicate_label_conflict'] = int(conflict.sum())
df = df[~conflict]

before = len(df)
df = df.drop_duplicates('md5', keep='first').reset_index(drop=True)
log['dropped_exact_duplicates'] = before - len(df)
pd.Series(log)

## 5. Group near-duplicates

Two images count as near-duplicates when their 64-bit pHashes differ in at most `NEAR` bits. Near-duplicates are kept but grouped, so a group never spans train and test. Groups that span two different Lumis classes are dropped as label conflicts.

In [ ]:
NEAR = 6

bits = np.array([[int(b) for b in bin(int(h, 16))[2:].zfill(64)] for h in df.phash], dtype=np.float32)
n = len(bits)
parent = list(range(n))

def find(i):
    while parent[i] != i:
        parent[i] = parent[parent[i]]
        i = parent[i]
    return i

pairs = []
inv = 1 - bits
for s in range(0, n, 1000):
    A = bits[s:s + 1000]
    dist = 64 - (A @ bits.T + (1 - A) @ inv.T)
    for i, j in zip(*np.where(dist <= NEAR)):
        gi = i + s
        if gi < j:
            pairs.append((gi, j, int(dist[i, j])))
            ri, rj = find(gi), find(j)
            if ri != rj:
                parent[rj] = ri

df['group_id'] = [find(i) for i in range(n)]
print('Near-duplicate pairs:', len(pairs))

### Review sample near-duplicate pairs

Check these look like the same photo. If many are clearly different photos, lower `NEAR` and rerun from step 5.

In [ ]:
import matplotlib.pyplot as plt
import random

random.seed(SEED)
sample = random.sample(pairs, min(10, len(pairs)))
if sample:
    fig, axes = plt.subplots(len(sample), 2, figsize=(6, 3 * len(sample)), squeeze=False)
    for r, (i, j, d) in enumerate(sample):
        for c, k in enumerate((i, j)):
            axes[r, c].imshow(Image.open(Path(LOCAL) / df.path[k]).convert('RGB'))
            axes[r, c].set_title(f'{df.lumis_class[k]}  (distance {d})', fontsize=8)
            axes[r, c].axis('off')
    plt.tight_layout()
    plt.savefig(f'{BASE}/exports/review_near_duplicates.jpg', dpi=80)
    plt.show()

In [ ]:
n_cls = df.groupby('group_id').lumis_class.nunique()
conflict = df.group_id.isin(n_cls[n_cls > 1].index)
log['dropped_near_duplicate_label_conflict'] = int(conflict.sum())
df = df[~conflict].reset_index(drop=True)

sizes = df.group_id.value_counts()
log['images_in_near_duplicate_groups'] = int(sizes[sizes > 1].sum())
log['final_images'] = len(df)
pd.Series(log)

## 6. Split 70/15/15 per class, keeping each near-duplicate group together

In [ ]:
import numpy as np
rng = np.random.default_rng(SEED)
group_split = {}
for cls, sub in df.groupby('lumis_class'):
    sizes = sub.groupby('group_id').size()
    order = rng.permutation(sizes.index.values)
    total, cum = sizes.sum(), 0
    for g in order:
        frac = cum / total
        group_split[g] = 'train' if frac < 0.70 else ('val' if frac < 0.85 else 'test')
        cum += sizes[g]
df['split'] = df.group_id.map(group_split)

table = pd.crosstab(df.lumis_class, df.split, margins=True, margins_name='total')[['train', 'val', 'test', 'total']]
table

## 7. Save the split, the cleaning log, and review sheets

In [ ]:
df.to_csv(SPLIT_FILE, index=False)
table.to_csv(f'{BASE}/exports/split_counts_v1.csv')
pd.Series(log, name='images').to_csv(f'{BASE}/exports/cleaning_log_v1.csv')

# One contact sheet per class from the training split, for a manual check of non-skin or watermarked images
for cls, sub in df[df.split == 'train'].groupby('lumis_class'):
    pick = sub.sample(min(48, len(sub)), random_state=SEED)
    fig, axes = plt.subplots(6, 8, figsize=(16, 12))
    for ax, p in zip(axes.flat, pick.path):
        ax.imshow(Image.open(Path(LOCAL) / p).convert('RGB'))
    for ax in axes.flat:
        ax.axis('off')
    fig.suptitle(f'{cls}: 48 random training images')
    plt.tight_layout()
    plt.savefig(f'{BASE}/exports/review_{cls}.jpg', dpi=60)
    plt.close()

print('Saved to Drive:')
print('  Lumis/data/lumis_split_v1.csv  (frozen split)')
print('  Lumis/exports/split_counts_v1.csv, cleaning_log_v1.csv')
print('  Lumis/exports/review_near_duplicates.jpg and review_<class>.jpg')